# Exploratory Data Analysis of the UCI Adult Census Income Dataset

**Data Science for Health Informatics — Homework 1**

This notebook presents an exploratory data analysis (EDA) of the **Adult Census Income dataset** from the UCI Machine Learning Repository. The analysis focuses on understanding the structure and quality of the data, preparing the data for analysis, and exploring relationships between demographic, educational, employment and income-related variables.

The dataset contains census information about individuals from the United States and is commonly used to study and predict whether an individual's annual income is above or below $50,000.


In [ ]:
student_name = "Chantale NzeggeMvele"
student_id = ""  # Student ID removed before public GitHub publication
student_background = "health"


## 1. Data Understanding

### 1.1 Dataset source

The dataset was obtained from the **UCI Machine Learning Repository**:

https://doi.org/10.24432/C5XW20


### 1.2 Domain area and purpose

The Adult dataset belongs mainly to the domains of **demography, economics, social science and data science**. The records are derived from U.S. Census Bureau data and contain demographic and employment characteristics. The dataset is commonly used for exploratory analysis and for classification tasks involving annual income.


### 1.3 Data file structure

The UCI distribution contains several files, including the training data (`adult.data`), test data (`adult.test`) and documentation files. The two data files are comma-separated and do not contain standard column headers. For this analysis, the training and test records are loaded separately, cleaned, and combined into one DataFrame. The documentation files are used for understanding the variables.


### 1.4 Number of features and observations

The combined dataset contains **48,842 observations and 15 columns**. There are 14 explanatory variables and one target variable, `income`. The explanatory variables describe demographic, educational, employment and financial characteristics of the individuals.


### 1.5 Numerical features

The dataset contains six numerical features: `age`, `fnlwgt`, `education_num`, `capital_gain`, `capital_loss`, and `hours_per_week`. These variables allow numerical analysis of age, sampling weight, education level, capital-related values and working hours.


### 1.6 Categorical features

There are eight categorical explanatory features: `workclass`, `education`, `marital_status`, `occupation`, `relationship`, `race`, `sex`, and `native_country`. Most are nominal variables. `education` has an ordered educational progression and can therefore be treated as ordinal when appropriate. The target variable, `income`, is also categorical.


### 1.7 Binary features

Two variables are binary in the dataset: `sex` has the categories **Female** and **Male**, while `income` has the categories **<=50K** and **>50K**. The income variable is the binary target used to distinguish the two income groups.


## 2. Basic Exploratory Data Analysis

### 2.1 Loading the dataset

The original training and test files are loaded from the UCI Machine Learning Repository. The test file contains a descriptive first line and income labels ending in a period, so these are handled before the two datasets are combined.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

column_names = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week",
    "native_country", "income"
]

train_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
test_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"

df_train = pd.read_csv(
    train_url,
    header=None,
    names=column_names,
    na_values=" ?",
    skipinitialspace=True
)

df_test = pd.read_csv(
    test_url,
    header=None,
    names=column_names,
    skipinitialspace=True
)

# Remove the descriptive first row from adult.test.
df_test = df_test.iloc[1:].copy()

# Remove the period used at the end of income labels in adult.test.
df_test["income"] = df_test["income"].str.replace(".", "", regex=False)

df = pd.concat([df_train, df_test], ignore_index=True)

df.head()


### 2.2 Data preprocessing pipeline

The preprocessing pipeline has four main purposes: select relevant features, assign appropriate data types, use clear and consistent feature names, and manage missing values. Because the objective of this homework is exploratory analysis, I retain the available variables rather than removing variables solely because they may be correlated.


In [ ]:
# Inspect the structure and missing values before preprocessing.
df.info()


### Feature selection

All 15 variables are retained because they describe different aspects of the population and can contribute to the exploratory questions. The `income` variable is retained as the target variable because several questions examine how demographic and employment characteristics relate to income.


In [ ]:
selected_columns = column_names
df = df[selected_columns].copy()


### Correct data types

The numerical variables are explicitly converted to numeric types. The demographic and employment variables, together with the income target, are converted to categorical data types. This makes the structure of the DataFrame clearer and supports efficient categorical analysis.


In [ ]:
numeric_columns = [
    "age", "fnlwgt", "education_num",
    "capital_gain", "capital_loss", "hours_per_week"
]

categorical_columns = [
    "workclass", "education", "marital_status", "occupation",
    "relationship", "race", "sex", "native_country", "income"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

for column in categorical_columns:
    df[column] = df[column].astype("category")

df.dtypes


### Renaming features

Two column names are renamed to make their meaning clearer. `fnlwgt` is changed to `final_weight`, and `education_num` is changed to `education_level`. The remaining names are already sufficiently descriptive for this analysis.


In [ ]:
df.rename(
    columns={
        "fnlwgt": "final_weight",
        "education_num": "education_level"
    },
    inplace=True
)

df.columns


### Missing values

Missing values are represented by `?` in the original data. They are converted to missing values during loading. For this exploratory analysis, rows containing missing values are removed because the analytical questions require complete observations across the variables being compared.


In [ ]:
# Check missing values.
df.isna().sum()


In [ ]:
df_clean = df.dropna().copy()

print("Original observations:", len(df))
print("Observations after removing missing values:", len(df_clean))
print("Removed observations:", len(df) - len(df_clean))

df_clean.head()


## 3. Univariate Analytical Questions

### Question 1: What is the average age of individuals who earn more than $50K?


In [ ]:
mean_age_high_income = (
    df_clean.loc[df_clean["income"] == ">50K", "age"]
    .mean()
)

print(f"Average age of individuals earning >50K: {mean_age_high_income:.2f} years")


**Reflection:** The average age of individuals earning more than $50K is approximately **44 years** in the cleaned dataset. This is older than the overall working-age population represented in the data. The result is plausible because higher income can be associated with accumulated work experience, although age alone does not explain income differences.


### Question 2: What is the median number of hours worked per week by women?


In [ ]:
median_hours_women = (
    df_clean.loc[df_clean["sex"] == "Female", "hours_per_week"]
    .median()
)

print(f"Median hours worked per week by women: {median_hours_women:.0f} hours")


**Reflection:** The median number of hours worked per week by women is **40 hours**, which corresponds to a conventional full-time working week. The median is useful here because it is less affected by people working unusually few or unusually many hours. The result provides a simple description of working patterns among women in the dataset.


## 4. Bivariate Analytical Questions

### Question 1: What is the average capital gain by education level?


In [ ]:
average_capital_gain = (
    df_clean.groupby("education", observed=True)["capital_gain"]
    .mean()
    .sort_values(ascending=False)
)

average_capital_gain


**Reflection:** The average capital gain differs substantially across education categories. Some of the highest averages occur in categories associated with higher educational attainment. However, this does not demonstrate that education causes capital gains. Capital gains are also influenced by factors such as wealth, investments, age and employment history.


### Question 2: How is income distributed across marital-status categories?


In [ ]:
income_by_marital_status = pd.crosstab(
    df_clean["marital_status"],
    df_clean["income"],
    normalize="index"
)

income_by_marital_status


**Reflection:** The income distribution varies across marital-status categories. Married-couple categories have a higher proportion of individuals earning above $50K than several unmarried categories. This association may reflect differences in age, employment, occupation or household circumstances, so marital status should not be interpreted as a direct cause of income.


## 5. Visualization-Based Analytical Questions

### Question 1: How are weekly working hours distributed across education levels?


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df_clean,
    x="education",
    y="hours_per_week"
)
plt.xticks(rotation=45, ha="right")
plt.title("Distribution of Hours Worked per Week by Education Level")
plt.xlabel("Education")
plt.ylabel("Hours Worked per Week")
plt.tight_layout()
plt.show()


**Reflection:** The boxplot shows that most education groups have weekly working hours concentrated around 40 hours, although the spread differs between groups. The visualization is useful because it shows the median, variability and potential outliers simultaneously. It also makes differences between education categories easier to compare than a table would.


### Question 2: How does age vary between the two income groups?


In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(
    data=df_clean,
    x="income",
    y="age"
)
plt.title("Age Distribution by Income Level")
plt.xlabel("Income")
plt.ylabel("Age")
plt.tight_layout()
plt.show()


**Reflection:** The boxplot indicates that the >50K income group tends to be older than the <=50K group, although there is substantial overlap. The visualization is useful because it clearly displays differences in the distributions rather than only comparing their average ages. The overlap also shows that age alone cannot determine income.


### Question 3: What is the relationship between education and the proportion of individuals in each income group?


In [ ]:
education_income = pd.crosstab(
    df_clean["education"],
    df_clean["income"],
    normalize="index"
)

education_income.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6)
)

plt.title("Income Proportion by Education Level")
plt.xlabel("Education")
plt.ylabel("Proportion")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


**Reflection:** The stacked bar chart shows a clear difference in income proportions across education categories. Higher educational attainment is generally associated with a larger proportion of individuals earning above $50K. The visualization is useful because normalization allows the income composition of groups with different sizes to be compared directly.


### Question 4: Is there a visible relationship between capital gain, capital loss and income group?


In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_clean,
    x="capital_gain",
    y="capital_loss",
    hue="income",
    alpha=0.6
)

plt.title("Capital Gain vs. Capital Loss by Income Level")
plt.xlabel("Capital Gain")
plt.ylabel("Capital Loss")
plt.tight_layout()
plt.show()


**Reflection:** Most observations are concentrated at zero for both capital gain and capital loss, which makes the relationship difficult to interpret from the scatterplot alone. Some observations with non-zero capital values belong to the higher-income group. The visualization is useful for identifying this concentration, but it is less effective for detailed comparison.


## 6. Pearson Correlation

### Question 9: Is there a linear correlation between age and hours worked per week?


In [ ]:
correlation = df_clean["age"].corr(
    df_clean["hours_per_week"],
    method="pearson"
)

print(f"Pearson correlation between age and hours worked per week: {correlation:.3f}")


**Reflection:** The Pearson correlation is approximately **0.10**, indicating a **weak positive linear relationship** between age and weekly working hours. The direction suggests that working hours tend to increase slightly with age, but the relationship is small. Therefore, age is not a strong linear predictor of weekly working hours in this dataset.


In [ ]:
plt.figure(figsize=(8, 6))
sns.regplot(
    data=df_clean,
    x="age",
    y="hours_per_week",
    scatter_kws={"alpha": 0.25}
)

plt.title("Age vs. Hours Worked per Week")
plt.xlabel("Age")
plt.ylabel("Hours Worked per Week")
plt.tight_layout()
plt.show()


## 7. Summary

This exploratory analysis demonstrates a basic data-science workflow: understanding the dataset before coding, loading and combining the source files, selecting and preparing variables, managing missing values, answering analytical questions, visualizing relationships, and interpreting numerical results.

The analysis suggests associations between income and variables such as age and education, while also showing that individual characteristics rarely explain income on their own. Further analysis could investigate these relationships using multivariable statistical or machine-learning methods.
